# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [4]:
import pandas as pd

In [5]:
#Definition of the tables.
data = [
    {
        "table": "employees",
        "definition": "Employee information, name"
    },
    {
        "table": "departments",
        "definition": "Department information, department name"
    }
]

df = pd.DataFrame(data)
print(df)

         table                               definition
0    employees               Employee information, name
1  departments  Department information, department name


In [6]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [7]:
print(text_tables)

employees: Employee information, name
departments: Department information, department name


In [8]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [10]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question="What are the names of the employees and the departments they work in?")

In [11]:
print(return_OAI(pqt1))

{
    "tables": ["employees", "departments"]
}


In [13]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question="Which tables should I use to answer questions about employee salaries?")

In [14]:
print(return_OAI(pqt3))

```json
{
    "tables": ["employees"]
}
```


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [15]:
# Definition of the tables
import pandas as pd

data = [
    {
        "table": "employees",
        "definition": "Employee master data including employee_id, first_name, last_name, department_id, hire_date and job_title_id."
    },
    {
        "table": "departments",
        "definition": "Department reference data including department_id, department_name and location."
    },
    {
        "table": "salaries",
        "definition": "Employee compensation data including employee_id, annual_salary, bonus and effective_date."
    },
    {
        "table": "job_titles",
        "definition": "Job title reference data including job_title_id, title_name, seniority_level and job_family."
    },
    {
        "table": "projects",
        "definition": "Project information including project_id, project_name, customer_id, start_date, end_date and project_type."
    },
    {
        "table": "project_assignments",
        "definition": "Bridge table connecting employees to projects including employee_id, project_id, role_on_project and allocation_percentage."
    },
    {
        "table": "customers",
        "definition": "Customer account data including customer_id, customer_name, industry and country."
    },
    {
        "table": "orders",
        "definition": "Sales order header data including order_id, customer_id, order_date, order_status and sales_region."
    },
    {
        "table": "order_items",
        "definition": "Sales order line items including order_id, product_id, quantity, unit_price and discount."
    },
    {
        "table": "products",
        "definition": "Product catalog data including product_id, product_name, product_category and list_price."
    }
]

df = pd.DataFrame(data)
df

,table,definition
0,employees,"Employee master data including employee_id, fi..."
1,departments,Department reference data including department...
2,salaries,Employee compensation data including employee_...
3,job_titles,Job title reference data including job_title_i...
4,projects,"Project information including project_id, proj..."
5,project_assignments,Bridge table connecting employees to projects ...
6,customers,"Customer account data including customer_id, c..."
7,orders,"Sales order header data including order_id, cu..."
8,order_items,"Sales order line items including order_id, pro..."
9,products,"Product catalog data including product_id, pro..."


In [16]:
prompt_question_tables = """
You are a senior data analyst. Your task is to select the minimum set of database tables needed to answer a user's SQL question.

Use only the table names and definitions provided below. Do not invent table names.

### Tables
{tables}

### User question
{question}

Return only valid JSON in this exact structure:
{{
  "tables": ["table_name_1", "table_name_2"],
  "reason": "Short explanation of why these tables are needed."
}}
"""

In [17]:
pqt1 = prompt_question_tables.format(
    tables=text_tables,
    question="List the first name, last name, department name, and annual salary of employees earning more than 60000."
)

print(pqt1)
print(return_OAI(pqt1))


You are a senior data analyst. Your task is to select the minimum set of database tables needed to answer a user's SQL question.

Use only the table names and definitions provided below. Do not invent table names.

### Tables
employees: Employee information, name
departments: Department information, department name

### User question
List the first name, last name, department name, and annual salary of employees earning more than 60000.

Return only valid JSON in this exact structure:
{
  "tables": ["table_name_1", "table_name_2"],
  "reason": "Short explanation of why these tables are needed."
}

{
  "tables": ["employees"],
  "reason": "The 'employees' table contains the first name, last name, and annual salary information of employees, which are needed to filter employees earning more than 60000."
}


In [18]:
pqt2 = prompt_question_tables.format(
    tables=text_tables,
    question="Calculate total revenue by product category for all completed orders placed in 2024."
)

print(pqt2)
print(return_OAI(pqt2))


You are a senior data analyst. Your task is to select the minimum set of database tables needed to answer a user's SQL question.

Use only the table names and definitions provided below. Do not invent table names.

### Tables
employees: Employee information, name
departments: Department information, department name

### User question
Calculate total revenue by product category for all completed orders placed in 2024.

Return only valid JSON in this exact structure:
{
  "tables": ["table_name_1", "table_name_2"],
  "reason": "Short explanation of why these tables are needed."
}

{
  "tables": ["departments"],
  "reason": "The 'departments' table is not directly related to the user question, but it is the only available table. Since the question does not involve product categories or orders, we can only provide the 'departments' table."
}


In [19]:
pqt3 = prompt_question_tables.format(
    tables=text_tables,
    question="Which employees worked on projects for the customer named Acme GmbH?"
)

print(pqt3)
print(return_OAI(pqt3))


You are a senior data analyst. Your task is to select the minimum set of database tables needed to answer a user's SQL question.

Use only the table names and definitions provided below. Do not invent table names.

### Tables
employees: Employee information, name
departments: Department information, department name

### User question
Which employees worked on projects for the customer named Acme GmbH?

Return only valid JSON in this exact structure:
{
  "tables": ["table_name_1", "table_name_2"],
  "reason": "Short explanation of why these tables are needed."
}

{
  "tables": ["employees", "departments"],
  "reason": "The 'employees' table is needed to retrieve employee information, including their names. The 'departments' table may be needed to link employees to projects and customers through department information."
}


In [20]:
pqt4 = prompt_question_tables.format(
    tables=text_tables,
    question="Which department has the highest average salary among employees assigned to AI projects?"
)

print(pqt4)
print(return_OAI(pqt4))


You are a senior data analyst. Your task is to select the minimum set of database tables needed to answer a user's SQL question.

Use only the table names and definitions provided below. Do not invent table names.

### Tables
employees: Employee information, name
departments: Department information, department name

### User question
Which department has the highest average salary among employees assigned to AI projects?

Return only valid JSON in this exact structure:
{
  "tables": ["table_name_1", "table_name_2"],
  "reason": "Short explanation of why these tables are needed."
}

{
  "tables": ["employees", "departments"],
  "reason": "We need the 'employees' table to access information about employees, including their assigned projects and salaries. We also need the 'departments' table to link employees to their respective departments."
}


Use this as your completed **Exercise** section. I also prepared a completed notebook here:

[Download completed SQL table-selection notebook](sandbox:/mnt/data/lab-sql-query-from-table-names-completed.ipynb)

## 1. Fix the table-definition cell

Replace your table-definition cell with this:

```python
# Definition of the tables
import pandas as pd

data = [
    {
        "table": "employees",
        "definition": "Employee master data including employee_id, first_name, last_name, department_id, hire_date and job_title_id."
    },
    {
        "table": "departments",
        "definition": "Department reference data including department_id, department_name and location."
    },
    {
        "table": "salaries",
        "definition": "Employee compensation data including employee_id, annual_salary, bonus and effective_date."
    },
    {
        "table": "job_titles",
        "definition": "Job title reference data including job_title_id, title_name, seniority_level and job_family."
    },
    {
        "table": "projects",
        "definition": "Project information including project_id, project_name, customer_id, start_date, end_date and project_type."
    },
    {
        "table": "project_assignments",
        "definition": "Bridge table connecting employees to projects including employee_id, project_id, role_on_project and allocation_percentage."
    },
    {
        "table": "customers",
        "definition": "Customer account data including customer_id, customer_name, industry and country."
    },
    {
        "table": "orders",
        "definition": "Sales order header data including order_id, customer_id, order_date, order_status and sales_region."
    },
    {
        "table": "order_items",
        "definition": "Sales order line items including order_id, product_id, quantity, unit_price and discount."
    },
    {
        "table": "products",
        "definition": "Product catalog data including product_id, product_name, product_category and list_price."
    }
]

df = pd.DataFrame(data)
df
```

---

## 2. Use this improved prompt template

```python
prompt_question_tables = """
You are a senior data analyst. Your task is to select the minimum set of database tables needed to answer a user's SQL question.

Use only the table names and definitions provided below. Do not invent table names.

### Tables
{tables}

### User question
{question}

Return only valid JSON in this exact structure:
{{
  "tables": ["table_name_1", "table_name_2"],
  "reason": "Short explanation of why these tables are needed."
}}
"""
```

---

## 3. Prompt Version 1: HR analytics

```python
pqt1 = prompt_question_tables.format(
    tables=text_tables,
    question="List the first name, last name, department name, and annual salary of employees earning more than 60000."
)

print(pqt1)
print(return_OAI(pqt1))
```

Expected tables:

```python
["employees", "departments", "salaries"]
```

---

## 4. Prompt Version 2: Sales analytics

```python
pqt2 = prompt_question_tables.format(
    tables=text_tables,
    question="Calculate total revenue by product category for all completed orders placed in 2024."
)

print(pqt2)
print(return_OAI(pqt2))
```

Expected tables:

```python
["orders", "order_items", "products"]
```

---

## 5. Prompt Version 3: Project/customer question

```python
pqt3 = prompt_question_tables.format(
    tables=text_tables,
    question="Which employees worked on projects for the customer named Acme GmbH?"
)

print(pqt3)
print(return_OAI(pqt3))
```

Expected tables:

```python
["employees", "project_assignments", "projects", "customers"]
```

This is a good example because the model must understand that `project_assignments` is the bridge table between employees and projects.

---

## 6. Prompt Version 4: Creative complex question

```python
pqt4 = prompt_question_tables.format(
    tables=text_tables,
    question="Which department has the highest average salary among employees assigned to AI projects?"
)

print(pqt4)
print(return_OAI(pqt4))
```

Expected tables:

```python
["departments", "employees", "salaries", "project_assignments", "projects"]
```

---

## 7. Weak prompt for comparison

Use this to show what does **not** work well:

```python
weak_prompt = f"""
Tables:
{text_tables}

Question:
Which department has the highest average salary among employees assigned to AI projects?

What should I use?
"""

print(return_OAI(weak_prompt))
```

This may return a long explanation, non-JSON output, or unnecessary tables.

---

# One-page report

## Objective

The objective of this lab was to test whether a language model can infer the correct database tables needed to answer a SQL question when it only receives table names and short table definitions. This is useful because real databases often contain many tables, and selecting the right tables is the first step before writing a correct SQL query.

## Prompt versions tested

I tested several prompt variations. The first version was a clear HR analytics question: finding employees, their departments, and their salaries. The expected tables were `employees`, `departments`, and `salaries`. This version worked well because the question directly mentioned employee information, department name, and salary.

The second version asked for revenue by product category for completed orders in 2024. The expected tables were `orders`, `order_items`, and `products`. This also worked well because revenue requires quantity and price from `order_items`, order date/status from `orders`, and product category from `products`.

The third version was more complex because it asked which employees worked on projects for a specific customer. This required a bridge table: `project_assignments`. The correct answer should include `employees`, `project_assignments`, `projects`, and `customers`. This was a good test because the model had to infer table relationships, not just match keywords.

The fourth version was the most creative: finding the department with the highest average salary among employees assigned to AI projects. This required combining HR data and project assignment data, so the expected tables were `departments`, `employees`, `salaries`, `project_assignments`, and `projects`.

## What worked well

The strongest prompt was the structured one that told the model to act as a senior data analyst, select the minimum set of tables, use only the provided table definitions, avoid inventing table names, and return valid JSON. This made the response easier to parse and more reliable. Asking for JSON was especially helpful because the output could be reused programmatically in Python.

## What did not work well

The weak prompt, which simply asked “What should I use?”, was less reliable. It could return long explanations, SQL examples, or tables that were not strictly necessary. In some cases, weak prompts can cause hallucination, where the model invents columns, joins, or table names that were not present in the provided schema.

## What I learned

I learned that prompt quality strongly affects the reliability of model output. A good prompt should include the role, task, table context, user question, constraints, and exact output format. For table selection tasks, the model performs better when it is explicitly told to choose the minimum required tables and not invent schema objects. I also learned that table definitions must be clear and specific. If table definitions are vague, the model may choose incorrect tables or miss bridge tables needed for joins.
